In [1]:
import os
import numpy as np

# Image segmentation
from scipy import ndimage

# Image processing and feature extraction modules
import skimage.measure as skm
import skimage as ski

ModuleNotFoundError: No module named 'skimage'

In [14]:
folder_path = "../../images/"
images_names = sorted([f for f in os.listdir(folder_path) if f.endswith('.png')])

images = {}

for image in images_names:
    full_path = os.path.join(folder_path + image)
    try:
        name_key = os.path.basename(full_path).split('.')[0]
        images[name_key] = ski.io.imread(full_path)
    except Exception as e:
        print(e)

## `extract_shape_feature` function

In [15]:
def extract_shape_features(img):
    img_filled = ndimage.binary_fill_holes(img)
    label_img = skm.label(img_filled, connectivity=img_filled.ndim)
    props = skm.regionprops(label_img)
    if len(props) == 0:
        return None
    main_lesion = max(props, key=lambda x: x.area)
    features = {
        "Area": main_lesion.area,
        "Area Bounding Box": main_lesion.area_bbox,
        "Area Convex": main_lesion.area_convex,
        "Area Filled": main_lesion.area_filled,
        "Axis Major Length": main_lesion.axis_major_length,
        "Axis Minor Length": main_lesion.axis_minor_length,
        "Centroid Local": main_lesion.centroid_local,
        "Eccentricity": main_lesion.eccentricity,
        "Extent": main_lesion.extent,
        "Equivalent Diameter": main_lesion.equivalent_diameter,
        "Euler Number": main_lesion.euler_number,
        "Feret Diameter Max": main_lesion.feret_diameter_max,
        "Perimeter": main_lesion.perimeter,
        "Solidity": main_lesion.solidity,

        # Manually calculated features as regionprops module does not support them

        "Roundness": 4*main_lesion.area / np.pi * main_lesion.axis_major_length**2,
        "Circularity": 4*np.pi*main_lesion.area/main_lesion.perimeter**2,
        "Aspect Ratio": main_lesion.axis_major_length / main_lesion.axis_minor_length,
        "Irregularity Index": main_lesion.perimeter / (2 * np.sqrt(np.pi * main_lesion.area))
    }
    
    return features

test_extract = extract_shape_features(images['1001'])
for i in test_extract.items():
    print(f'{i[0]} : {i[1]}')


Area : 126898.0
Area Bounding Box : 194000.0
Area Convex : 152510.0
Area Filled : 126898.0
Axis Major Length : 450.1080207470438
Axis Minor Length : 377.94048854056
Centroid Local : [224.40123564 181.57650239]
Eccentricity : 0.5431028292457931
Extent : 0.6541134020618556
Equivalent Diameter : 401.9596394512796
Euler Number : 1
Feret Diameter Max : 500.75642781695774
Perimeter : 3675.119333432866
Solidity : 0.832063471247787
Roundness : 32733948885.97349
Circularity : 0.11806515740691106
Aspect Ratio : 1.1909494600199175
Irregularity Index : 2.9103091502764067
